# Aula 8 — HTML, `requests` e BeautifulSoup

Até agora a turma coletou dado pronto: exportação do Zeeschuimer, CSV já organizado. Hoje a fonte é uma página da internet, sem exportação nenhuma: você pede a página pro servidor, ela chega como um textão de HTML, e cabe a você achar as partes que interessam e organizar isso numa tabela.

Isso é **web scraping estático**: baixar o HTML de uma página (sem rodar o JavaScript dela, sem clicar em nada) e extrair conteúdo de dentro dele. Funciona para sites que já mandam o conteúdo pronto no HTML (blogs, catálogos, listas). Páginas que montam o conteúdo depois, via JavaScript, no navegador, precisam de outra ferramenta (Playwright, Aula 9). Hoje é só o caso estático, que já cobre bastante coisa.

Interessa porque boa parte do dado que existe na internet não tem API, não tem exportação, só existe como página. Scraping é o jeito de transformar "página visível pra gente" em "tabela que dá pra analisar".

## 1. Revisão rápida

Antes de seguir, uma passada rápida no que já vimos e que volta hoje: URL (o endereço de uma página), JSON e CSV (formatos de dado que vamos gerar no fim), scripts Python rodados com `uv run`, e leitura de mensagem de erro (vamos precisar de novo, porque scraping quebra com frequência: página muda, busca no HTML não bate, conexão cai). Se algo disso soa vago, vale espiar as Aulas 1 e 2 rapidinho.


## 2. HTML: do que uma página é feita

HTML (HyperText Markup Language) é o esqueleto de qualquer página. Cada pedaço de conteúdo fica dentro de uma **tag**, escrita entre `<` e `>`:

```html
<h1>Título da página</h1>
<p>Um parágrafo qualquer.</p>
```

Tags têm início (`<p>`) e fim (`</p>`), e o que fica entre elas é o conteúdo. Tags também podem ter **atributos**, que dão informação extra sobre aquele elemento:

```html
<a href="https://exemplo.com" class="link-post" id="post-1">Texto do link</a>
```

Aqui, `<a>` é a tag (define um link), `href` é o atributo com o endereço de destino, `class` é um atributo usado pra agrupar elementos parecidos (vários posts podem ter `class="link-post"`), e `id` é um atributo que identifica **um único** elemento na página (não repete).

HTML é uma **árvore**: toda tag pode conter outras tags dentro dela (filhas), e estar dentro de outra tag (mãe). Por exemplo:

```html
<ul class="lista-posts">
    <li class="post">
        <h2 class="titulo-post">Título do post</h2>
        <p class="autor-post">Autor: Fulano</p>
    </li>
    <li class="post">
        ...
    </li>
</ul>
```

`<ul>` é mãe de dois `<li>`, e cada `<li>` é mãe de um `<h2>` e um `<p>`. Pensar em "onde na árvore está o dado que eu quero" é metade do trabalho de scraping; a outra metade é escrever o código que chega lá.

## 3. Tag, classe e id: como apontar pro elemento certo

Pra achar um pedaço do HTML, o BeautifulSoup precisa de uma descrição do elemento. Nesta disciplina a gente faz isso com `.find()` / `.find_all()`, passando:

| O que você passa | O que pega | Exemplo |
|---|---|---|
| Nome da tag | todo elemento daquele tipo | `sopa.find_all("p")` pega todo `<p>` |
| `{"class": "..."}` | elemento com aquela classe | `sopa.find("li", {"class": "post"})` pega o primeiro `<li class="post">` |
| `id="..."` | o único elemento com aquele id | `sopa.find(id="titulo-site")` pega o elemento com `id="titulo-site"` |

O filtro de atributos é um **dicionário**, o mesmo tipo de dado que vocês já usam o semestre inteiro: `{"class": "post"}` diz "quero a classe post".

Dá pra combinar tag + classe (como na tabela) e, quando o elemento que você quer está *dentro* de outro, você faz em dois passos: primeiro acha o bloco maior, depois procura dentro dele. Isso aparece na Seção 10.


## 4. Antes de escrever código: inspecione a página no navegador

Aqui vai o passo mais importante da aula, e o mais fácil de pular: **não adivinhe a tag/classe/id, olhe no navegador**.

Com qualquer página aberta no Chrome, Firefox ou Edge:

1. Clique com o botão direito em cima do elemento que te interessa (o título de um post, por exemplo) e escolha **"Inspecionar"** (ou "Inspecionar elemento").
2. Isso abre o painel de **DevTools**, na aba **Elements** (Chrome/Edge) ou **Inspecionar** (Firefox), já com aquele trecho de HTML destacado.
3. Também dá pra abrir o painel direto, sem clicar em nada específico: `Ctrl+Shift+I` no Windows, `Cmd+Option+I` no Mac. Ou `F12` na maioria dos navegadores.
4. No painel, passe o mouse pelas tags: cada uma que você passa fica destacada na página ao vivo. Isso confirma se aquele `<h2 class="titulo-post">` é mesmo o elemento que você queria.
5. Anote a **tag**, a **classe** e o **id** (se tiver). São exatamente os argumentos que você vai passar pro `.find()` / `.find_all()` daqui a pouco.

**OBS:** o HTML que aparece no DevTools às vezes já foi alterado por JavaScript depois que a página carregou, então pode ser levemente diferente do que `requests` vai te devolver (que é o HTML "cru", antes do JavaScript rodar). Para páginas estáticas como as de hoje isso quase nunca é problema, mas é bom saber que existe: se o que você anotou bater no navegador e não bater no código, essa é uma causa provável.


## 5. Requisição HTTP: pedir a página pro servidor

Quando você digita um endereço no navegador, ele faz uma **requisição HTTP** pro servidor daquele site, pedindo o conteúdo. O tipo mais comum de requisição é **GET**: "me devolve esse recurso", sem enviar nem alterar nada no servidor (é o que scraping usa quase sempre).

O servidor responde com um **status code**, um número que resume o que aconteceu:

| Código | Significa |
|---|---|
| `200` | Deu certo, aqui está o conteúdo |
| `301` / `302` | A página mudou de endereço (redirecionamento) |
| `403` | Proibido, o servidor recusou a te atender |
| `404` | Não encontrado, esse endereço não existe |
| `429` | Você pediu demais, o servidor está te limitando |
| `500` | Erro do lado do servidor, não seu |

Checar o status **antes** de tentar extrair qualquer coisa evita um erro confuso mais na frente: se a página não veio (`404`, por exemplo), tentar procurar um `<h2>` nela não vai achar nada, e não é porque a busca no HTML está errada.


## 6. A biblioteca `requests`

`requests` é a biblioteca padrão do Python pra fazer requisições HTTP. Ela não vem pronta no Python, por isso entra no `requirements.txt` da raiz (junto com `beautifulsoup4`, que usamos daqui a pouco).

**Não crie um `.venv` novo dentro desta pasta da aula.** Use o ambiente da raiz do repositório.

No terminal (o integrado do VS Code funciona bem aqui), **dentro da pasta da disciplina**:

```cmd
uv pip install -r requirements.txt
```

Se o `uv` não funcionar:

```cmd
pip install -r requirements.txt
```

Os mesmos comandos funcionam no Mac (Terminal).

O padrão básico de uso é sempre o mesmo: fazer o `GET`, checar `.status_code`, e só então usar `.text` (o HTML como string). Vamos ver isso primeiro com um arquivo local (100% previsível, sem depender de internet em aula) e depois repetir com uma URL real.


## 7. Passo 1: a página de teste local

Antes de sair batendo em sites de verdade, vamos praticar com `exemplos/pagina-teste.html`: um blog de exemplo, com título e seis posts (cada um com título, link, autor, data e resumo). Ela não vem de uma requisição (é um arquivo já salvo no seu computador), mas o HTML dentro dela é igualzinho ao que uma página real te devolveria, e assim conseguimos praticar `BeautifulSoup` sem depender da internet estar funcionando.


In [ ]:
from pathlib import Path  # Path deixa lidar com caminho de arquivo de forma mais simples

caminho_pagina_teste = Path("exemplos/pagina-teste.html")  # onde está a página de teste
html_local = caminho_pagina_teste.read_text(encoding="utf-8")  # lê o arquivo inteiro como uma string de texto

print(f"Página lida: {len(html_local)} caracteres")  # confirma que o arquivo carregou e mostra o tamanho
print(html_local[:300])  # espia os primeiros 300 caracteres, só pra ver que é HTML mesmo

## 8. Parseando o HTML com BeautifulSoup

Ter o HTML como string (o que fizemos acima) não ajuda muito sozinho: procurar texto dentro dele "na unha" seria lento e cheio de erro. O `BeautifulSoup` transforma essa string numa árvore navegável, a mesma árvore que vimos na Seção 2, e dá métodos prontos para procurar dentro dela.

In [ ]:
from bs4 import BeautifulSoup  # importa a classe principal da biblioteca

sopa = BeautifulSoup(html_local, "html.parser")  # transforma a string de HTML numa árvore navegável
# "html.parser" é o parser (interpretador de HTML) que já vem com o Python, sem precisar instalar mais nada

print(sopa.title.get_text())  # .title pega a tag <title> da página, .get_text() pega só o texto de dentro dela

## 9. `.find()` e `.find_all()`: achando elementos

`.find()` devolve o **primeiro** elemento que bate com o que você pediu (ou `None`, se não achar nada). `.find_all()` devolve **todos**, numa lista. Os dois aceitam o nome da tag e, opcionalmente, filtros extras, em geral como um dicionário de atributos (os mesmos `class` e `id` que você anotou no DevTools na Seção 4).


In [ ]:
primeiro_post = sopa.find("li", {"class": "post"})  # acha o primeiro <li class="post"> da árvore
print(primeiro_post.find("h2").get_text())  # dentro dele, acha o <h2> e pega o texto

todos_os_posts = sopa.find_all("li", {"class": "post"})  # agora acha TODOS os <li class="post">
print(f"Total de posts encontrados: {len(todos_os_posts)}")  # confirma quantos vieram


## 10. Buscando por id e dentro de um elemento

Além de tag + classe, `.find()` aceita `id="..."` quando o elemento tem um identificador único. E, quando o pedaço que você quer está *dentro* de outro (um link dentro de um post, por exemplo), o caminho é em dois passos: primeiro acha o bloco maior, depois chama `.find()` **nesse** elemento, não na sopa inteira.


In [ ]:
titulo_do_site = sopa.find(id="titulo-site")  # o único elemento com esse id
print(titulo_do_site.get_text())

posts = sopa.find_all("li", {"class": "post"})  # todos os <li class="post">
print(f"Total de posts (via .find_all): {len(posts)}")

primeiro_post = sopa.find("li", {"class": "post"})  # o primeiro post
primeiro_titulo = primeiro_post.find("h2", {"class": "titulo-post"})  # o h2 de título DENTRO desse post
print(primeiro_titulo.get_text())


## 11. Extraindo texto e atributos

`.get_text()` pega o texto visível de dentro de um elemento (e de tudo que está dentro dele). Para pegar um **atributo** (como o `href` de um link), trata-se o elemento quase como um dicionário: `elemento["href"]`.

**ATENÇÃO:** texto extraído de HTML às vezes vem com espaços ou quebras de linha sobrando nas pontas (indentação do próprio arquivo HTML). `.get_text(strip=True)` já remove isso; se você usar só `.get_text()`, vale aplicar `.strip()` depois.


In [ ]:
primeiro_post = sopa.find("li", {"class": "post"})  # começa pelo bloco do post
link_do_primeiro_post = primeiro_post.find("a", {"class": "link-post"})  # o link dentro desse post

texto_do_link = link_do_primeiro_post.get_text(strip=True)  # texto visível do link, sem espaços nas pontas
endereco_do_link = link_do_primeiro_post["href"]  # atributo href: o endereço de destino

print(f"Texto: {texto_do_link}")
print(f"Link: {endereco_do_link}")


## 12. De vários elementos para uma lista de dicionários

Isso é o coração do scraping: percorrer todos os itens (aqui, cada `<li class="post">`) e, para cada um, montar um dicionário com os campos que interessam. No fim, sobra uma lista de dicionários, o mesmo formato que já usamos em CSV nas Aulas 4 e 5.


In [ ]:
posts_extraidos = []  # lista vazia que vai guardar um dicionário por post

for post in sopa.find_all("li", {"class": "post"}):  # percorre cada <li class="post"> da página
    titulo = post.find("h2", {"class": "titulo-post"}).get_text(strip=True)  # título do post
    link = post.find("a", {"class": "link-post"})["href"]  # endereço do link
    autor = post.find("p", {"class": "autor-post"}).get_text(strip=True)  # linha "Autor: ..."
    data = post.find("p", {"class": "data-post"}).get_text(strip=True)  # linha "Publicado em: ..."

    posts_extraidos.append({  # guarda tudo num dicionário e adiciona na lista
        "titulo": titulo,
        "link": link,
        "autor": autor,
        "data": data,
    })

print(f"Total extraído: {len(posts_extraidos)}")
posts_extraidos[:2]  # espia os dois primeiros, pra conferir o formato


## 13. Salvando em CSV

Com a lista de dicionários pronta, salvar em CSV é o mesmo `csv.DictWriter` já usado em aulas anteriores: primeiro escreve o cabeçalho (nomes das colunas), depois uma linha por dicionário.


In [ ]:
import csv  # módulo padrão do Python para ler e escrever CSV
from pathlib import Path

Path("dados").mkdir(exist_ok=True)  # garante que a pasta dados/ existe antes de salvar

with open("dados/pagina-teste.csv", "w", encoding="utf-8", newline="") as arquivo:  # abre o arquivo de saída
    colunas = ["titulo", "link", "autor", "data"]  # nomes das colunas, na ordem que queremos no CSV
    escritor = csv.DictWriter(arquivo, fieldnames=colunas)  # cria o escritor, avisando quais colunas existem
    escritor.writeheader()  # escreve a primeira linha, com os nomes das colunas
    escritor.writerows(posts_extraidos)  # escreve uma linha por dicionário da lista

print("Salvo em dados/pagina-teste.csv")

## 14. Passo 2: repetindo com uma página real

Agora o mesmo fluxo (requisitar → checar status → parsear → extrair → salvar), só que numa URL de verdade. Duas páginas feitas justamente para prática de scraping, com permissão explícita para isso, e por isso boas escolhas para treinar sem preocupação ética:

- **http://books.toscrape.com**: catálogo fake de livros, com título, preço e link de cada um.
- **http://quotes.toscrape.com**: lista de citações, com texto, autor e tags.

Vamos usar `books.toscrape.com` aqui. Repare que o código muda muito pouco em relação ao que já fizemos: troca a origem do HTML (de um `Path.read_text()` para um `requests.get()`), e o resto do fluxo (BeautifulSoup, `.find()` / `.find_all()`, lista de dicionários, CSV) é o mesmo.


In [ ]:
import requests  # biblioteca de requisição HTTP

url = "http://books.toscrape.com/"
resposta = requests.get(url)  # faz a requisição GET

print(f"Status: {resposta.status_code}")  # sempre confira o status antes de seguir

if resposta.status_code == 200:  # só segue se a página veio certinho
    resposta.encoding = resposta.apparent_encoding  # corrige a codificação de texto (evita "Â£" no lugar de "£")
    html_real = resposta.text  # .text é o HTML como string, igual ao que lemos do arquivo local
    print(f"HTML recebido: {len(html_real)} caracteres")
else:
    html_real = None
    print("A página não respondeu como esperado, confira a URL e a conexão antes de continuar.")

**O que observar:** o padrão `if resposta.status_code == 200:` é o que vale guardar desta aula. Nunca parseie ou extraia nada de uma resposta sem checar o status antes, é exatamente o critério que o exercício de hoje vai cobrar.

In [ ]:
livros_extraidos = []  # mesma ideia da Seção 12, agora para os livros do site real

if html_real:  # só roda se a requisição anterior deu certo
    sopa_real = BeautifulSoup(html_real, "html.parser")  # parseia o HTML recebido

    for produto in sopa_real.find_all("article", {"class": "product_pod"}):  # cada livro fica num <article class="product_pod">
        link_titulo = produto.find("h3").find("a")  # a tag <a> dentro do <h3>; o título completo está no atributo title
        titulo = link_titulo["title"]  # o texto visível vem cortado com "...", o atributo title tem o título inteiro
        link_relativo = link_titulo["href"]  # o href vem relativo (sem o "http://books.toscrape.com/" na frente)
        link_completo = requests.compat.urljoin(url, link_relativo)  # junta com a URL base pra virar um link completo
        preco = produto.find("p", {"class": "price_color"}).get_text(strip=True)  # preço do livro

        livros_extraidos.append({
            "titulo": titulo,
            "preco": preco,
            "link": link_completo,
        })

print(f"Total extraído: {len(livros_extraidos)}")
livros_extraidos[:3]  # espia os três primeiros


In [ ]:
if livros_extraidos:  # só salva se a extração acima rendeu algo
    with open("dados/books-toscrape.csv", "w", encoding="utf-8", newline="") as arquivo:
        colunas = ["titulo", "preco", "link"]
        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()
        escritor.writerows(livros_extraidos)
    print("Salvo em dados/books-toscrape.csv")
else:
    print("Nada para salvar (a requisição real pode ter falhado, veja a Seção 14).")

## 15. Limites, `robots.txt` e boas práticas

Só porque uma página está pública, não quer dizer que dá pra raspar do jeito que der. Antes de coletar de um site novo:

- **Confira o `robots.txt`:** todo site pode publicar, no endereço `site.com/robots.txt`, quais partes ele permite ou não que robôs (inclusive scripts de scraping) acessem. Por exemplo, [books.toscrape.com/robots.txt](http://books.toscrape.com/robots.txt). Nem todo site respeita isso via lei, mas é a convenção da internet para declarar intenção, e ignorar um `Disallow` explícito é malvisto (e, em alguns contextos, pode ter consequência legal).
- **Termos de uso da página:** `robots.txt` é técnico; o Termos de Uso do site é o texto que efetivamente diz o que pode ou não. Sites de prática como os de hoje já autorizam isso explicitamente.
- **Não sobrecarregue o servidor:** um script mal escrito pode disparar centenas de requisições por segundo sem querer. Se for coletar várias páginas, espace as requisições com `time.sleep(1)` (ou mais) entre uma e outra.
- **Identifique-se com um User-Agent:** por padrão, `requests` já manda um User-Agent genérico, mas alguns sites preferem (ou exigem) que você se identifique. Dá pra customizar assim:

```python
cabecalhos = {"User-Agent": "projeto-aula-fgv (contato: seu-email@exemplo.com)"}
resposta = requests.get(url, headers=cabecalhos)
```

- **Dado pessoal continua sendo dado pessoal:** os mesmos limites éticos discutidos na Aula 4 (sobre redes sociais) valem aqui. Só porque dá pra técnicamente extrair, não quer dizer que deveria.

## 16. Quando der errado

- **`ConnectionError`:** o `requests` não conseguiu nem chegar no servidor. Confira sua conexão de internet e se a URL está digitada certo (não esqueça o `http://` ou `https://`).
- **Status diferente de `200`:** confira o número no dicionário da Seção 5. Um `404` costuma ser URL errada; um `403` ou `429` pode ser o site bloqueando esse tipo de acesso, revise se aquela coleta é mesmo permitida antes de insistir.
- **`.find()` retornando `None`, e depois um `AttributeError`:** isso acontece quando a busca não achou nada (a página mudou, a tag/classe/id tem um erro de digitação, ou o conteúdo só aparece depois de rodar JavaScript, que `requests` não executa). Sempre que uma busca for nova, teste ela sozinha numa célula (`print(sopa.find("h2", {"class": "titulo-post"}))`) antes de usar dentro de um loop.
- **Texto com espaços ou `\n` estranhos:** use `.get_text(strip=True)` em vez de só `.get_text()`, como fizemos na Seção 11.
- **Acento ou símbolo estranho (tipo `Â£` no lugar de `£`):** o `requests` às vezes erra a codificação de texto ao adivinhar. Ajuste com `resposta.encoding = resposta.apparent_encoding` antes de ler `.text`, como fizemos na Seção 14.
- **`ModuleNotFoundError: No module named 'bs4'` (ou `'requests'`):** as dependências não foram instaladas nesse ambiente. Na pasta da disciplina, rode `uv pip install -r requirements.txt` de novo (Seção 6).


## Prática: faça agora

Abra `exercicios/exercicio-08-html-requests-beautifulsoup.ipynb`. Ele pede a extração de uma página real permitida (books.toscrape.com, quotes.toscrape.com, ou outra que você tenha certeza que pode raspar), seguindo o mesmo fluxo de hoje: requisitar, checar o status, inspecionar no navegador antes de escrever o `.find()` / `.find_all()`, extrair os campos, conferir e salvar em CSV com colunas nomeadas.
